In [ ]:
"""
Sistema de Alertas para Workflows
"""

import threading
from datetime import datetime
from typing import Dict, List, Optional, Any, Callable
from dataclasses import dataclass, field
from enum import Enum
import json
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False


class AlertLevel(Enum):
    """Níveis de alerta"""
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"
    CRITICAL = "critical"


@dataclass
class Alert:
    """Representa um alerta"""
    level: AlertLevel
    message: str
    timestamp: datetime = field(default_factory=datetime.utcnow)
    step_name: Optional[str] = None
    workflow_name: Optional[str] = None
    details: Dict[str, Any] = field(default_factory=dict)
    
    def to_dict(self) -> dict:
        return {
            "level": self.level.value,
            "message": self.message,
            "timestamp": self.timestamp.isoformat(),
            "step_name": self.step_name,
            "workflow_name": self.workflow_name,
            "details": self.details
        }


@dataclass
class AlertConfig:
    """Configuração de alertas"""
    # Slack
    slack_webhook_url: Optional[str] = None
    slack_channel: Optional[str] = None
    
    # Email
    smtp_host: Optional[str] = None
    smtp_port: int = 587
    smtp_user: Optional[str] = None
    smtp_password: Optional[str] = None
    email_from: Optional[str] = None
    email_to: Optional[List[str]] = None
    
    # Teams
    teams_webhook_url: Optional[str] = None
    
    # Throttling
    throttle_seconds: int = 60  # Mínimo entre alertas iguais
    max_alerts_per_minute: int = 10


class AlertManager:
    """
    Gerenciador central de alertas
    
    Suporta múltiplos canais (log, Slack, Email, Teams) com
    throttling para evitar spam.
    """
    
    _instance = None
    _lock = threading.Lock()
    
    def __init__(self, config: AlertConfig = None):
        self.config = config or AlertConfig()
        self._alert_history: List[Alert] = []
        self._last_alert_time: Dict[str, datetime] = {}
        self._alerts_this_minute: int = 0
        self._minute_start: datetime = datetime.utcnow()
        self._data_lock = threading.Lock()
        
        # Handlers customizados
        self._custom_handlers: Dict[str, Callable[[Alert], None]] = {}
    
    @classmethod
    def get_instance(cls) -> 'AlertManager':
        """Retorna a instância singleton"""
        with cls._lock:
            if cls._instance is None:
                cls._instance = cls()
            return cls._instance
    
    @classmethod
    def configure(cls, config: AlertConfig):
        """Configura o AlertManager com novas configurações"""
        with cls._lock:
            cls._instance = cls(config)
    
    def register_handler(self, name: str, handler: Callable[[Alert], None]):
        """Registra um handler customizado de alertas"""
        self._custom_handlers[name] = handler
    
    def _should_throttle(self, alert_key: str) -> bool:
        """Verifica se o alerta deve ser throttled"""
        now = datetime.utcnow()
        
        # Verificar limite por minuto
        if (now - self._minute_start).total_seconds() > 60:
            self._minute_start = now
            self._alerts_this_minute = 0
        
        if self._alerts_this_minute >= self.config.max_alerts_per_minute:
            return True
        
        # Verificar throttle individual
        if alert_key in self._last_alert_time:
            elapsed = (now - self._last_alert_time[alert_key]).total_seconds()
            if elapsed < self.config.throttle_seconds:
                return True
        
        return False
    
    def send_alert(
        self,
        level: str,
        message: str,
        channels: List[str] = None,
        step_name: str = None,
        workflow_name: str = None,
        details: Dict = None,
        force: bool = False
    ):
        """
        Envia um alerta através dos canais especificados
        
        Args:
            level: Nível do alerta (info, warning, error, critical)
            message: Mensagem do alerta
            channels: Lista de canais (default: ["log"])
            step_name: Nome do step que gerou o alerta
            workflow_name: Nome do workflow
            details: Detalhes adicionais
            force: Se True, ignora throttling
        """
        channels = channels or ["log"]
        
        alert = Alert(
            level=AlertLevel(level),
            message=message,
            step_name=step_name,
            workflow_name=workflow_name,
            details=details or {}
        )
        
        # Gerar chave única para throttling
        alert_key = f"{level}:{step_name}:{message[:50]}"
        
        with self._data_lock:
            # Verificar throttling
            if not force and self._should_throttle(alert_key):
                return
            
            self._last_alert_time[alert_key] = datetime.utcnow()
            self._alerts_this_minute += 1
            self._alert_history.append(alert)
        
        # Enviar para cada canal
        for channel in channels:
            try:
                self._send_to_channel(channel, alert)
            except Exception as e:
                # Log do erro mas não falhar
                from .logger import get_logger
                logger = get_logger("alerts")
                logger.error(f"Falha ao enviar alerta para {channel}: {e}")
    
    def _send_to_channel(self, channel: str, alert: Alert):
        """Envia alerta para um canal específico"""
        handlers = {
            "log": self._send_to_log,
            "slack": self._send_to_slack,
            "email": self._send_to_email,
            "teams": self._send_to_teams,
        }
        
        if channel in handlers:
            handlers[channel](alert)
        elif channel in self._custom_handlers:
            self._custom_handlers[channel](alert)
        else:
            from .logger import get_logger
            get_logger("alerts").warning(f"Canal desconhecido: {channel}")
    
    def _send_to_log(self, alert: Alert):
        """Envia alerta para log"""
        from .logger import get_logger
        logger = get_logger("alerts")
        
        log_methods = {
            AlertLevel.INFO: logger.info,
            AlertLevel.WARNING: logger.warning,
            AlertLevel.ERROR: logger.error,
            AlertLevel.CRITICAL: logger.critical,
        }
        
        log_method = log_methods.get(alert.level, logger.info)
        log_method(f"[ALERTA] {alert.message}", **alert.details)
    
    def _send_to_slack(self, alert: Alert):
        """Envia alerta para Slack"""
        if not self.config.slack_webhook_url:
            return
        
        if not HAS_REQUESTS:
            from .logger import get_logger
            get_logger("alerts").warning("requests não instalado, Slack desabilitado")
            return
        
        color_map = {
            AlertLevel.INFO: "#36a64f",
            AlertLevel.WARNING: "#ffcc00",
            AlertLevel.ERROR: "#ff6600",
            AlertLevel.CRITICAL: "#ff0000",
        }
        
        payload = {
            "attachments": [{
                "color": color_map.get(alert.level, "#808080"),
                "title": f"[{alert.level.value.upper()}] Self-Healing Workflow Alert",
                "text": alert.message,
                "fields": [],
                "footer": f"Workflow: {alert.workflow_name or 'N/A'} | Step: {alert.step_name or 'N/A'}",
                "ts": int(alert.timestamp.timestamp())
            }]
        }
        
        if alert.details:
            for key, value in alert.details.items():
                if key != "traceback":
                    payload["attachments"][0]["fields"].append({
                        "title": key,
                        "value": str(value)[:200],
                        "short": True
                    })
        
        if self.config.slack_channel:
            payload["channel"] = self.config.slack_channel
        
        requests.post(
            self.config.slack_webhook_url,
            json=payload,
            timeout=10
        )
    
    def _send_to_email(self, alert: Alert):
        """Envia alerta por email"""
        cfg = self.config
        if not all([cfg.smtp_host, cfg.email_from, cfg.email_to]):
            return
        
        msg = MIMEMultipart()
        msg['From'] = cfg.email_from
        msg['To'] = ', '.join(cfg.email_to)
        msg['Subject'] = f"[{alert.level.value.upper()}] Workflow Alert: {alert.message[:50]}"
        
        body = f"""
        <h2>Self-Healing Workflow Alert</h2>
        <p><strong>Level:</strong> {alert.level.value}</p>
        <p><strong>Message:</strong> {alert.message}</p>
        <p><strong>Workflow:</strong> {alert.workflow_name or 'N/A'}</p>
        <p><strong>Step:</strong> {alert.step_name or 'N/A'}</p>
        <p><strong>Time:</strong> {alert.timestamp.isoformat()}</p>
        """
        
        if alert.details:
            body += "<h3>Details:</h3><pre>"
            body += json.dumps(alert.details, indent=2, default=str)
            body += "</pre>"
        
        msg.attach(MIMEText(body, 'html'))
        
        with smtplib.SMTP(cfg.smtp_host, cfg.smtp_port) as server:
            if cfg.smtp_user and cfg.smtp_password:
                server.starttls()
                server.login(cfg.smtp_user, cfg.smtp_password)
            server.send_message(msg)
    
    def _send_to_teams(self, alert: Alert):
        """Envia alerta para Microsoft Teams"""
        if not self.config.teams_webhook_url:
            return
        
        if not HAS_REQUESTS:
            return
        
        color_map = {
            AlertLevel.INFO: "00FF00",
            AlertLevel.WARNING: "FFFF00",
            AlertLevel.ERROR: "FFA500",
            AlertLevel.CRITICAL: "FF0000",
        }
        
        payload = {
            "@type": "MessageCard",
            "@context": "http://schema.org/extensions",
            "themeColor": color_map.get(alert.level, "808080"),
            "summary": f"Workflow Alert: {alert.message[:50]}",
            "sections": [{
                "activityTitle": f"[{alert.level.value.upper()}] Self-Healing Workflow Alert",
                "facts": [
                    {"name": "Message", "value": alert.message},
                    {"name": "Workflow", "value": alert.workflow_name or "N/A"},
                    {"name": "Step", "value": alert.step_name or "N/A"},
                    {"name": "Time", "value": alert.timestamp.isoformat()},
                ],
                "markdown": True
            }]
        }
        
        requests.post(
            self.config.teams_webhook_url,
            json=payload,
            timeout=10
        )
    
    def get_alert_history(self, limit: int = 100) -> List[Dict]:
        """Retorna histórico de alertas"""
        with self._data_lock:
            recent = sorted(
                self._alert_history,
                key=lambda a: a.timestamp,
                reverse=True
            )[:limit]
            return [a.to_dict() for a in recent]
    
    def clear_history(self):
        """Limpa histórico de alertas"""
        with self._data_lock:
            self._alert_history.clear()
            self._last_alert_time.clear()


def send_alert(
    level: str,
    message: str,
    channels: List[str] = None,
    **kwargs
):
    """Função auxiliar para enviar alertas rapidamente"""
    AlertManager.get_instance().send_alert(
        level=level,
        message=message,
        channels=channels,
        **kwargs
    )
